## Support Vector Machines (SVM): Walking through the Math in Detail
#### Binary SVM Classification: The Set-Up
- Input: $x \in \mathbb{R}^d$
- Target: $y \in \{+1, -1 \}$
- Objective: find a $d-1$ dimensional hyperplane
$w^\top x + b$<br>
such that it <strong>maximizes the margin between the two classes</strong>.

The idea is that if the classes are linearly separable, i.e., $\exists \: w$ and $b$ such that

$y_i(w^\top x_i + b) > 0, \:\:\forall i$<br>

This means the hyperplane separates the training points so that one class falls onto one side of the plane, and the other class falls onto the other side.<br><br>

Then, assuming linear separability, since the training set is finite, we can take the minimum over all training points $x_i$ so that<br>

$m = \text{min}_{x} \left[ y (w^\top x + b) \right]$<br>

Let $x_j$ be where that minimum occurs, i.e.,

$x_j = \text{argmin}_{x} \left[ y (w^\top T x + b) \right]$<br>

then we rescale w and b

$w \leftarrow \text{sign}(y_j) \frac{w}{m}$<br>
$b \leftarrow \text{sign}(y_j) \frac{b}{m}$<br>

so that

$w^\top x_j + b = 1$

And for all other training points $x_i$ belonging to the same class as $x_j$,

$w^\top x_i + b >= 1$<br><br>

Similarly on the other side of the hyperplane (i.e., the other class different than $x_j$), with the newly scaled $w$ and $b$, it must be that 

$w^\top x_i + b \le -1$ 

for all $x_i$ on that side. So points on that side must fall on or outside of the plane $w^\top x + b = -1$ As such, the two hyperplanes that seperate the classes

$w^\top x + b = 1$<br>
$w^\top x + b = -1$

are $\frac{2}{\left\lVert w \right\lVert} = \frac{2}{w^\top w}$ apart. This is called the _hard-margin SVM classifier_, which aims to maximize that distance, which is equivalent to minimizing $\left\lVert w \right\lVert$.<br><br>

So in optimization theory notation, the hard margin optimization problem is

$\text{minimize}_{w,b} \quad \frac{\left\Vert w \right\Vert^2}{2}$<br>
$\text{subject to} \quad y_i(w^\top x_i + b) \ge 1, \:\:\forall i$<br><br>

Now this is where $C$ comes in. The _soft-margin SVM classifier_ allows for points to fall in the gap between the hyperplanes $w^\top x + b = 1$ and $w^\top x + b = -1$. So we introduce violations $\xi_i$ such that, instead of requiring $y_i (w^\top x_i + b) \ge 1, \forall i$, we allow some "slack"

$y_i (w^\top x_i + b) \ge 1 - \xi_i, \:\:\forall i$<br><br>

But then we need to add a <strong>penalty term</strong> to what we want to minimize:

$\text{minimize}_{w,b,ξ} \quad \frac{\left\Vert w \right\Vert ^2}{2} + C\sum_i ξ_i$

Meaning the loss has to balance the <strong>trade-off between maximizing margin and minimizing the violations $\xi$.</strong><br><br>

So now the optimization problem for the _soft-margin SVM classifier_ becomes

$\text{minimize}_{w,b,ξ} \quad \frac{\left\Vert w \right\Vert ^2}{2} + C\sum_i \xi_i$<br>
$\text{subject to} \quad y_i (w^\top x_i + b) \ge 1 - \xi_i, \:\:\forall i$<br><br>

Note that, by re-arranging the constraints, we have $\xi_i \ge 1 - y_i (w^\top x_i + b)$, so for any given $w$ and $b$, minimizing $C\sum_i \xi_i$ means taking the <i>hinge loss</i>

$\xi_i = \text{max} \left[0, 1 - y_i (w^\top x_i + b)\right]$

That is, if the point correctly classified and satisfies the constraint, $\xi_i = 0$; if the point is correctly classified but violates the constraint, $0 < \xi_i \le 1$; if $\xi_i > 1$, wrong classification.<br><br>

Moreover, if $C \rightarrow \infty$, the model is pushed to find $w$ such that $\xi_i$ are as close to zero as possible, and the constraint becomes closer and closer to the hard margin constraint has the $\xi_i →0$ (if linearly separable). This means the model will have less tolerance for training set variability — which leads to violations — and will find wildly different $w$ across different training sets, i.e., this is <strong>high variance situation with low regularization</strong>.

On the other hand, if $C = 0$, the model is not penalized for the violations, so it doesn't get "punished" for just choosing $w = 0$ whatever the training set looks like. That is what <strong>bias</strong> looks like and the model is not classifying at all, hence an overly regularized and underfitting story. That's why $C$ is the regularization parameter that is <i>inversely proportional to regularization strength</i>.

#### Lagrangian, Primal, and Dual
Recall that, for soft margin SVM, the <strong>primal minimization problem</strong> is

$\text{minimize}_{w,b,ξ} \quad \frac{\left\Vert w \right\Vert ^2}{2} + C\sum_i \xi_i$<br>
$\text{subject to} \quad y_i (w^\top x_i + b) \ge 1 - \xi_i, \:\:\forall i$<br><br>

Re-arranging the constraint, we have $g_i(w, b, \xi) = 1 - y_i (w^\top x_i + b) - \xi_i \le 0$ and $h_i(\xi) = -\xi_i \le 0$. So we introduce Lagrange multipliers $\alpha_i \ge 0$ and $\beta_i \ge 0$ for each constraint and build the Lagrangian

$\mathcal{L}(w, b ,\xi, \alpha, \beta; D) = f(w, \xi; D) + \sum_i \alpha_i g_i(w, b, \xi; D) + \sum_i \beta_i h_i(\xi; D) \quad$ where $\xi = (\xi_i, \dots, \xi_n)$<br>
> Note that the optimiazation problem is optimizing over $w$, $b$, and $\xi$. The training set $D = \{(x_i, y_i)\}_{i=1}^n$ are treated as constants that the optimization problem is conditional on.

Plugging in the functions explicitly, we have

$\mathcal{L}(w, b, \xi, \alpha, \beta) = \frac{\left\Vert w \right\Vert ^2}{2} + C\sum_i \xi_i + \sum_i \alpha_i (1 - y_i (w^\top x_i + b) - \xi_i) - \sum_i \beta_i \xi_i$<br><br>

Now, for <strong>the dual problem</strong>, we take the infimum of the Lagrangian over the primal variables, then take the supremum over the multipliers

$\text{sup}_{\alpha, \beta \ge 0}\: q(\alpha, \beta) = \text{sup}_{\alpha, \beta \ge 0}\: \text{inf}_{w, b, \xi}\: \mathcal{L}\: (w, b, \xi, \alpha, \beta)$<br>

where the dual function is defined as<br>

$q(\alpha, \beta) = \text{inf}_{w, b, \xi}\: \mathcal{L}\: (w, b, \xi, \alpha, \beta)$<br>

In other words, for each fixed set of multipliers, we find the lower bound of the Lagrangian, then we take the greatest (i.e., tightest) lower bound.
> Note that for each fixed set of multipliers $\{\alpha, \beta\} = \{(\alpha_i, \beta_i)\}_{i=1}^n$, any <i>feasible (i.e., satisfying the constraints) set of variables</i> $\{w, b, \xi\}$ gives $\sum_i \alpha_i g_i \le 0$ and $\sum_i \beta_i h_i \le 0$, and therefore<br><br>$\mathcal{L}(w, b, \xi, \alpha, \beta) \le f(w, \xi) \quad \forall \:\: \text{feasible} \:\{w, b, \xi\}$<br><br>That is, let $p^*$ be the primal optimum, then we have<br><br>$q(\alpha, \beta) = \text{inf}_{w, b, \xi}\: \mathcal{L}\: (w, b, \xi, \alpha, \beta) \le \mathcal{L}\: (w, b, \xi, \alpha, \beta) \le p^*$

To evaluate the infimum, we require that the gradient of the Lagrangian be zero (<strong>stationarity condition</strong>), i.e.,

$\nabla_{w,b,\xi}\:\mathcal{L} = 0$<br><br>
$\frac{\partial \mathcal{L}}{\partial w} = w - \sum_i \alpha_i y_i x_i = 0 \:\:\Longrightarrow \: w =  \sum_i \alpha_i y_i x_i$<br><br>
$\frac{\partial \mathcal{L}}{\partial b} = -\sum_i \alpha_i y_i = 0$<br><br>
$\frac{\partial \mathcal{L}}{\partial xi_i} = C - \alpha_i - \beta_i = 0 \:\: \Longrightarrow \: 0 \le \alpha_i \le C \:\:$ since $\beta_i \ge 0$, i.e., the multipliers $\alpha_i$ are <i>boxed</i> by $C$.<br><br>
Plug these back into $q(\alpha, \beta)$ and taking the supremum

$\frac{\left\Vert w \right\Vert ^2}{2} = \frac{1}{2} w^\top w = \frac{1}{2} \sum_i \sum_j \alpha_i \alpha_j y_i y_j x^{\top}_{i} x_j$<br><br>
$\sum_i \alpha_i (1 - y_i (w^\top x_i + b) - \xi_i)$<br>
$= \sum_i \alpha_i - \sum_i \sum_j \alpha_i \alpha_j y_i y_j x^{\top}_{i} x_j - b\sum_i \alpha_i y_i - \sum_i \alpha_i \xi_i$<br>
$= \sum_i \alpha_i - \sum_i \sum_j \alpha_i \alpha_j y_i y_j x^{\top}_{i} x_j - \sum_i \alpha_i \xi_i$<br><br>
$q(\alpha, \beta)$<br>
$= \frac{1}{2} \sum_i \sum_j \alpha_i \alpha_j y_i y_j x^{\top}_{i} x_j + C\sum_i \xi_i + \sum_i \alpha_i - \sum_i \sum_j \alpha_i \alpha_j y_i y_j x^{\top}_{i} x_j - \sum_i \alpha_i \xi_i - \sum_i \beta_i \xi_i$<br>
$= \sum_i \alpha_i - \frac{1}{2} \sum_i \sum_j \alpha_i \alpha_j y_i y_j x^{\top}_{i} x_j$<br>

That is, the dual problem is to find the supremum (we'll drop the $\beta$ since $q(.)$ is no longer dependent on it)

$\text{sup}_{\alpha}\:\: q(\alpha) = \text{sup}_{\alpha}\: \sum_i \alpha_i - \frac{1}{2} \sum_i \sum_j \alpha_i \alpha_j y_i y_j x^{\top}_{i} x_j$<br>
$\text{subject to} \quad \sum_i \alpha_i y_i = 0, \quad 0 \le \alpha_i \le C \:\:\: \forall \:i$<br><br>

According to the <strong>complementary slackness</strong> condition in KKT, we need, at optimum ${w^*, b^*, \xi_*}$

$\alpha_i (1 - y_i (w^{*\top} x_i + b^*) - \xi^*_i) = 0$

and 

$\beta_i \xi^*_i = (C - \alpha_i) \xi^*_i = 0\:\:\:$ (the second equality is from the stationarity condition)

for each $i$.

We devide the points into three cases: strictly outside the margin (correctly classified with no violations), strictly within the margin (correctly classified but with violations or misclassified), and exactly on the margin boundary

- For points <strong>strictly outside the margin</strong>, $y_i (w^{*\top} x_i + b^*) > 1$, so $1 - y_i (w^{*\top} x_i + b^*) - \xi^*_i < 0$ since $\xi^*_i \geq 0$. This forces $\alpha_i = 0$, which in turn forces $\xi^*_i = 0$ to satisfy $(C - \alpha_i) \xi^*_i = 0$.
- For points <strong>strictly within the margin</strong>, $y_i (w^{*\top} x_i + b^*) < 1$, so $\xi^*_i \geq 1 - y_i (w^{*\top} x_i + b^*) > 0$ by the primal constraint, which forces $\alpha_i = C$.
- For points <strong>excatly on the margin boundary</strong>, $y_i (w^{*\top} x_i + b^*) = 1$, so the complementary slackness conditions become
    - $\alpha_i \xi^*_i = 0$
    - $(C - \alpha_i) \xi^*_i = 0$<br>The only way to satisfy both, since $0 \leq \alpha_i \leq C$ is $\xi^*_i = 0$, and $\alpha_i$ can be anywhere inside the box $\left[0, C\right]$.<br>
    
<strong>In sum, when</strong> $\alpha_i > 0$, <strong>the point is either strictly within the margin or on the margin boundary. These points are called the support vectors</strong>.

> Note that we can compress the <strong>quadratic form</strong> $\text{sup}_{\alpha, \beta \ge 0}\: \sum_i \alpha_i - \frac{1}{2} \sum_i \sum_j \alpha_i \alpha_j y_i y_j x^{\top}_{i} x_j$ in matrix notation:<br><br>$\mathbf{1}^\top \mathbf{\alpha} - \frac{1}{2} \mathbf{\alpha}^\top \mathbf{H} \mathbf{\alpha}$<br><br>where<br><br>$\mathbf{H} = \mathbf{Z} \mathbf{Z}^\top , \:\: \mathbf{Z}_{i} = x_i y_i$.

#### The Kernel Trick
Note that in the dual

$q(\alpha) = \sum_i \alpha_i - \frac{1}{2} \sum_i \sum_j \alpha_i \alpha_j y_i y_j x^{\top}_{i} x_j$

the training points $x_i$ only appear in pairs in the dot products $x^{\top}_{i} x_j$. If we replace them with a <strong>kernel function</strong>

$K(x_i, x_j) := \phi(x_i)^\top \phi(x_j)$<br>
$q(\alpha) = \sum_i \alpha_i - \frac{1}{2} \sum_i \sum_j \alpha_i \alpha_j y_i y_j K(x_i, x_j)$<br>

then this allows us to introduce non-linearity. Though not explicitly computed, $\phi(.)$ maps vectors in the original input space into another feature space -- possibly infinite dimensional -- endowed with an inner product, where the kernel function computes the dot product between the mapped vectors. Thus, an SVM with a non-linear kernel is <strong>linear in the mapped vector space, but not linear in the original input space</strong>. This is the kernel trick.


#### Support Vector Machine for Regression (SVR): The Set-up
- Input: $x \in \mathbb{R}^d$
- Target: $y \in \mathbb{R}$
- Objective: find an affine linear map that predicts the target from the input
$\hat{y} = f(x) = w^\top x + b$<br>
where $w \in \mathbb{R}^d$ and $b \in \mathbb{R}$.

However, SVR differs from OLS (ordinary least squares) in that, instead of computing the squared residual $(y_i - \hat{y_i})^2$ for <i>every training point</i>, SVR has a tolerance range for small prediction erros, and the loss $L_{\epsilon}$ is zero if the residual is less than $\epsilon$ for some $\epsilon > 0$:

$|r_i| = |y_i - \hat{y_i}| = |y_i - f(x_i)| = |y_i - (w^\top x + b)|$<br>
$L_{\epsilon}(y_i, f(x_i)) := \text{max}(0, |r_i| - \epsilon) = \text{max}(0, |y_i - f(x_i)| - \epsilon)$<br>

That is, geometrically, we can think of a band/tube around the function $(f(x) - \epsilon, f(x) + \epsilon)$ where points that fall inside this <i>epsilon tube</i> contribute nothing to the loss. This is called $\epsilon$<strong>-insensitive</strong>.<br><br>

Now suppose in an ideal world, all training points $(x_i, y_i)$ fall inside the epsilon tube -- in which case, the loss is effectively zero for all $i$. We then ask, of all such affine linear maps $w^\top x + b$ (i.e., every point is inside the epsilon tube), what is the <i>flattest</i> function? That is, we try to solve the following constrained optimization problem

$\text{minimize}_{w, b} \quad \frac{1}{2} \left\Vert w \right\Vert^2$<br>
$\text{subject to} \quad |y_i - (w^\top x_i + b)| \le \epsilon, \:\:\: \forall \: i$<br>

This is the "hard-margin" SVR analogy.<br><br>

As with soft-margin SVM, we add slack variables $\xi_i \ge 0$ and $\xi^*_i \ge 0$, $i = 1, \dots, n$ (one for each side of the absolute value inequality) to account for points that don't fall inside the epsilon tube

$y_i - (w^\top x_i + b) \le \xi_i + \epsilon, \:\: \forall i$<br>
$(w^\top x_i + b) - y_i \le \xi^*_i + \epsilon, \:\: \forall i$<br>

and add a penalty term so that the constrained optimization becomes

$\text{minimize}_{w,b,\xi,\xi^*} \quad \frac{1}{2} \left\Vert w \right\Vert^2 + C \sum_i (\xi_i + \xi^*_i)$<br>
$\text{subject to} \quad \: y_i - (w^\top x_i + b) \le \xi_i + \epsilon, \quad (w^\top x_i + b) - y_i \le \xi^*_i + \epsilon, \quad \xi_i \ge 0, \quad \xi^*_i \ge 0, \:\:\: \forall i$

Again, this means the model balances the trade-off between violations (inside the epsilon tube or not) and function flatness (the flatter the function, the less steep the slope -- in 1D analogy -- and hence the less sensitive the function is to input changes). That is, if $C$ is large, violations are expensive,




#### Lagrangian, Primal, and Dual: Again, Why They're Called "Support Vectors"
To build the Lagrangian, we need for multipliers $\alpha \ge 0$, $\alpha^* \ge 0$, $\eta \ge 0$, and $\eta^* \ge 0$.

$\mathcal{L}(w,b,\alpha,\alpha^*,\eta,\eta^*)$<br>
$= \frac{1}{2} \left\Vert w \right\Vert^2 + C \sum_i (\xi_i + \xi^*_i)$<br>
$\:\:\:\:+ \sum_i \alpha_i (y_i - (w^\top x_i + b) - \xi_i - \epsilon) + \sum_i \alpha^*_i ((w^\top x_i +b) - y_i - \xi^*_i - \epsilon) - \sum_i \eta_i \xi_i - \sum_i \eta^* \xi^*_i$<br>

Differentiating $\mathcal{L}$ w.r.t the primal variables

$\frac{\partial \mathcal{L}}{\partial w} = w - \sum_i \alpha_i x_i + \sum_i \alpha^*_i x_i = 0 \Longrightarrow w = \sum_i (\alpha_i - \alpha^*_i) x_i$<br><br>
$\frac{\partial \mathcal{L}}{\partial b} = \sum_i \alpha_i - \sum_i \alpha^*_i = 0 \Longrightarrow \sum_i (\alpha_i - \alpha^*_i) = 0$<br><br>
$\frac{\partial \mathcal{L}}{\partial \xi_i} = C - \alpha_i - \eta_i = 0 \Longrightarrow 0 \le \alpha_i \le C$<br><br>
$\frac{\partial \mathcal{L}}{\partial \xi^*_i} = C - \alpha^*_i - \eta^*_i = 0 \Longrightarrow 0 \le \alpha^*_i \le C$<br><br>

Plugging them back in the Lagrangian, then we have the dual

$q(\alpha, \alpha^*, \eta, \eta^*) = \sum_i y_i (\alpha_i - \alpha^*_i) - \epsilon \sum_i (\alpha_i + \alpha^*_i) - \frac{1}{2}\sum_i \sum_j (\alpha_i - \alpha^*_i)(\alpha_j - \alpha^*_j)x_i^\top x_j$<br>

Dropping $\eta, \eta^*$ and taking the supremum, we have

$\text{sup}_{\alpha, \alpha^*} \: q(\alpha, \alpha^*) = \text{sup}_{\alpha, \alpha^*} \: \sum_i y_i (\alpha_i - \alpha^*_i) - \epsilon \sum_i (\alpha_i + \alpha^*_i) - \frac{1}{2}\sum_i \sum_j (\alpha_i - \alpha^*_i)(\alpha_j - \alpha^*_j)x_i^\top x_j$<br>
$\text{subject to} \quad \sum_i (\alpha_i - \alpha^*_i) = 0, \quad 0 \le \alpha_i \le C \quad 0 \le \alpha^*_i \le C, \:\:\: \forall \: i$<br><br>

According to <strong>complementary slackness</strong>,

$\alpha_i (y_i - (w^\top x_i + b) - \xi_i - \epsilon) = 0$<br>
$\alpha^*_i ((w^\top x_i + b) - y_i - \xi^*_i - \epsilon) = 0$<br>
$\eta_i \xi_i = (C - \alpha_i) \xi_i = 0$<br>
$\eta^*_i \xi^*_i = (C - \alpha^*_i) \xi^*_i = 0$<br>

for all $i$.

- For points strictly inside the epsilon tube, $|y_i - (w^\top x_i) + b| < \epsilon$, so $|y_i - (w^\top x_i) + b| - \tilde{\xi_i} - \epsilon < 0$ since $\tilde{\xi_i} \ge 0, \:\: \tilde{\xi_i} = \xi_i, \: \xi^*_i$. Therefore, $\alpha_i = \alpha^*_i = 0$.
- For points strictly outside the epsilon tube, either
    - $y_i - (w^\top x_i) + b - \epsilon > 0$,  or
    - $(w^\top x_i) - y_i + b - \epsilon > 0$
    
    and therefore either
    - $\xi_i \ge y_i - (w^\top x_i + b) - \epsilon > 0$, or
    - $\xi^*_i \ge (w^\top x_i + b) - y_i - \epsilon > 0$

    by the primal constraints, which forces either $\alpha_i = C$ or $\alpha^*_i = C$.
- For points exactly on the epsilon tube boundary, either
    - $y_i - (w^\top x_i) + b - \epsilon = 0$,  or
    - $(w^\top x_i) - y_i + b - \epsilon = 0$

    this forces either $\xi_i = 0$ or $\xi^*_i = 0$, and $\alpha_i \in [0, C]$ (or $\alpha^*_i$).

That is, <strong>only points outside or on the tube boudnary can have non-zero multipliers. Just like the SVM case, these are the support vectors.</strong>

In [201]:
import os
import numpy as np
import pandas as pd
import sklearn
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.model_selection import (train_test_split,
                                     GridSearchCV,
                                     RandomizedSearchCV,
                                     PredefinedSplit,
                                     RepeatedKFold,
                                     RepeatedStratifiedKFold,
                                     LeaveOneOut,
                                     KFold,
                                     StratifiedKFold,
                                     BaseCrossValidator)
from sklearn.feature_selection import SelectKBest, SelectPercentile, f_regression
from sklearn.metrics import r2_score, mean_absolute_error, make_scorer, check_scoring
from sklearn.metrics._scorer import _Scorer
from sklearn.svm import SVR, LinearSVR, SVC, LinearSVC
import seaborn as sns
import matplotlib.pyplot as plt
from collections import namedtuple
from collections.abc import Iterable
from typing import Callable
from types import FunctionType
from IPython.display import clear_output
import time

DATA_DIR = "/Users/jowanglin/brainhack-ntu/machine-learning/nilearn_data"

feat_file_64 = f"{DATA_DIR}/MAIN_BASC064_subsamp_features.npz"
feat_file_07 = f"{DATA_DIR}/MAIN_BASC007_subsamp_features.npz"
pheno_file = f"{DATA_DIR}/pheno.xlsx"
try:
    X = np.load(feat_file_64)["a"]
    X_07 = np.load(feat_file_07)["a"]
    pheno = pd.read_excel(pheno_file)
    y = pheno["Age"]
    age_group = pheno["AgeGroup"]
except FileNotFoundError as e:
    print(e)
else:
    print(X.shape)
    display(pheno)

(155, 2016)


,participant_id,Age,AgeGroup,Child_Adult,Gender,Handedness
0,sub-pixar155,26.00,Adult,adult,M,R
1,sub-pixar123,27.06,Adult,adult,F,R
2,sub-pixar124,33.44,Adult,adult,M,R
3,sub-pixar125,31.00,Adult,adult,M,R
4,sub-pixar126,19.00,Adult,adult,F,R
...,...,...,...,...,...,...
150,sub-pixar054,5.99,5yo,child,M,R
151,sub-pixar055,5.82,5yo,child,M,L
152,sub-pixar056,5.26,5yo,child,F,R
153,sub-pixar049,5.23,5yo,child,M,R


In [131]:
RANDOM_STATE = 42

def check_cv_instance(cv):
     if isinstance(cv, int):
        if cv < 2:
            raise ValueError ("For integer cv, must be >= 2.")
        print("""cv passed as an integer. Note that by default sklearn will NOT shuffle the folds. If your data are ordered (e.g., sorted by target value),
pass cv as a BaseCrossValidator object an set shuffle=True.""")
        return True
     elif isinstance(cv, BaseCrossValidator):
        return True # accepts cv as KFold, StratifiedKFold, etc.
     elif isinstance(cv, Iterable):
        print("cv appears to be an iterable. Assuming it is a valid iterable of (train_index, test_index) splits...")
        return True
     else:
         return False

def tune_hyperparams(estimator, estimator_name: str,
                   X_fit, y_fit, *,
                   cv: int | Iterable | BaseCrossValidator | None=None, ps_test_fold:list | None=None,
                   search: str="grid", params_space: dict | None=None,
                   refit: bool=False, scoring: str | list | Callable | None=None,
                   n_iter: int=200, n_jobs: int=-1,
                   **steps):
    pipe_steps = [(k, v) for k, v in steps.items()] + [(estimator_name, estimator)]
    pipe = Pipeline(steps=pipe_steps)

    if search == "grid":
        searcher = GridSearchCV(estimator=pipe,
                            scoring=scoring,
                            refit=refit, 
                            param_grid=params_space,
                            n_jobs=n_jobs,
                            return_train_score=True)
    elif search == "rand":
        searcher = RandomizedSearchCV(estimator=pipe,
                                      scoring=scoring,
                                      refit=refit, 
                                      param_distributions=params_space,
                                      n_jobs=n_jobs,
                                      n_iter=n_iter,
                                      random_state=RANDOM_STATE,
                                      return_train_score=True)
    
    # use cross validation for inner loop to tune hyperparams
    if check_cv_instance(cv):
        searcher.set_params(cv=cv)
         
    # use a fixed heldout validation set in the inner loop to tune hyperparams
    elif cv is None:
        if ps_test_fold is None:
                raise Exception ("Please provide explicit indices for the predefined splitter")
        ps = PredefinedSplit(test_fold=ps_test_fold)
        searcher.set_params(cv=ps)
        
    else:
         raise TypeError ("Please pass cv as an integer > 1 for cross validation or pass cv=None for a fixed held-out validation set in the inner loop.")
        
    searcher.fit(X_fit, y_fit)
    
    return searcher


In [194]:
NestedCVResults = namedtuple("NestedCV", ["searcher", "outer_cv_scores", "outer_oof_predictions", "params_sets"])

def instantiate_estimator(estimator_name: str, **estimator_params):
    if estimator_params is None:
        estimator_params = {}
    if estimator_name == "svr":
        kernel = estimator_params.get("kernel", "linear")
        estimator = SVR(kernel=kernel)
    elif estimator_name == "svc":
        kernel = estimator_params.get("kernel", "linear")
        estimator = SVC(kernel=kernel)
    return estimator

def nested_cv(X_train_outer, y_train_outer, *,
              estimator_name: str, estimator_params: dict | None=None,
              stratify: bool=True, n_repeats: None | int=1, outer_cv: str | int=5,
              inner_cv: Iterable | BaseCrossValidator | int=5, search: str="grid", params_space: dict | None=None,
              inner_scoring: str | list | Callable | None=None, outer_scoring: str | list | Callable | None=None,
              n_iter: int=200, n_jobs: int=-1,
              **steps):
    if not isinstance(X_train_outer, np.ndarray):
        X_train_outer = np.array(X_train_outer)
    if not isinstance(y_train_outer, np.ndarray):
        y_train_outer = np.array(y_train_outer)
        
    if estimator_params is None:
            estimator_params = {}

    if outer_cv == "loo":
        splitter = LeaveOneOut()
    
    elif isinstance(outer_cv, int) and outer_cv > 1:
        if stratify:
            # stratify = True for classification
            splitter = RepeatedStratifiedKFold(n_repeats=n_repeats, n_splits=outer_cv, random_state=RANDOM_STATE)
        else:
            # stratify = False for regression
            splitter = RepeatedKFold(n_repeats=n_repeats, n_splits=outer_cv, random_state=RANDOM_STATE)

    searchers, outer_cv_scores, params_sets = [], [], []

    if outer_cv == "loo" or n_repeats == 1: 
        outer_oof_predictions = {1: np.empty(y_train_outer.shape, dtype=y_train_outer.dtype)}
    else:
        outer_oof_predictions = {i+1: np.empty(y_train_outer.shape, dtype=y_train_outer.dtype)
                                                                    for i in range(n_repeats)}
    
    for i, (fit_idx, valid_idx) in enumerate(splitter.split(X_train_outer, y_train_outer)):
        display_outer_cv = len(y_train_outer) if outer_cv == "loo" else outer_cv
        print(f"REPEAT {1 + i // display_outer_cv}; FOLD {1 + i % display_outer_cv}")
        
        X_fit, X_valid = X_train_outer[fit_idx], X_train_outer[valid_idx]
        y_fit, y_valid = y_train_outer[fit_idx], y_train_outer[valid_idx] 
        
        estimator_inner = instantiate_estimator(estimator_name, **estimator_params)
        searcher = tune_hyperparams(
                               estimator_inner, estimator_name,
                               X_fit, y_fit, 
                               cv=inner_cv,
                               search=search,
                               params_space=params_space,
                               refit=False,
                               scoring=inner_scoring,
                               n_iter=n_iter,
                               n_jobs=n_jobs,
                               **steps
                               )
        searchers.append(searcher)
        
        estimator_outer = instantiate_estimator(estimator_name, **estimator_params)
        
        pipe_steps = [(k, v) for k, v in steps.items()] + [(estimator_name, estimator_outer)]
        pipe = Pipeline(steps=pipe_steps)
        pipe.set_params(**searcher.best_params_)
        pipe.fit(X_fit, y_fit)

        y_pred = pipe.predict(X_valid)
        outer_oof_predictions[1 + i // display_outer_cv][valid_idx] = y_pred

        if isinstance(outer_scoring, FunctionType):
            outer_scoring = make_scorer(outer_scoring) 
        elif isinstance(outer_scoring, list):
            outer_scoring = [make_scorer(s) if isinstance(s, FunctionType) else s for s in outer_scoring]
            
        scoring = check_scoring(pipe, scoring=outer_scoring, allow_none=False)
        score = scoring(pipe, X_valid, y_valid)
        
        outer_cv_scores.append(score)
        params_sets.append(searcher.best_params_)
    
    results = NestedCVResults(searchers, outer_cv_scores, outer_oof_predictions, params_sets)
    return results


In [149]:
TEST_SIZE = 0.15
HELDOUT_SIZE = 0.1765
# -> train : valid : test = 70% : 15% : 15%

# HYPERPARAMS = [PERCENTILE, C, EPSILON, GAMMA]
k_dist = list(range(50, 401))
c_dist = np.logspace(-3, 3, 100) # 1e-3 -> 1e3
epsilon_dist = np.linspace(0.001, 0.2, 100)
gamma_dist = np.logspace(-4, 1, 20) 

percentile_grid = list(range(5, 26)) # 5 % -> 25%
c_grid = np.logspace(-2, 2, 20) # 1e-2 -> 1e2
epsilon_grid = np.linspace(0.001, 0.2, 20)
gamma_grid = np.logspace(-4, 1, 10)

X_fit, X_test, y_fit, y_test = train_test_split(X, y,
                                                test_size=TEST_SIZE,
                                                shuffle=True,
                                                stratify=pheno["AgeGroup"],
                                                random_state=RANDOM_STATE)

X_train_inner, X_heldout, y_train_inner, y_heldout = train_test_split(X_fit,
                                                y_fit,
                                                test_size=HELDOUT_SIZE,
                                                shuffle=True,
                                                stratify=pheno.loc[y_fit.index,"AgeGroup"],
                                                random_state=RANDOM_STATE)

ps_test_fold = np.where(y_fit.index.isin(y_heldout.index), 0, -1)
searcher1  = tune_hyperparams(SVR(kernel="linear"), "svr_linear_kernel",
                           X_fit, y_fit, 
                           ps_test_fold=ps_test_fold,
                           cv=None,
                           search="rand", params_space={"univariate_best_k__k": k_dist,
                                                       "svr_linear_kernel__C": c_dist,
                                                       "svr_linear_kernel__epsilon": epsilon_dist},
                            refit=True,
                            scoring=None,
                            univariate_best_k=SelectKBest(f_regression)
                            )
best_estimator1 = searcher1.best_estimator_
print("**Fixed Held-Out Validation Set w/ Best-K Selector (F-Regression) Using Randomized Search**")
print(f"""Best C: {round(searcher1.best_params_['svr_linear_kernel__C'], 6)}
Best epsilon: {round(searcher1.best_params_['svr_linear_kernel__epsilon'], 6)}""")
selector = best_estimator1.named_steps["univariate_best_k"]
print(f"Best K: {selector.k}")
print(f">> Test R^2 = {round(best_estimator1.score(X_test, y_test), 6)}")

**Fixed Held-Out Validation Set w/ Best-K Selector (F-Regression) Using Randomized Search**
Best C: 1.417474
Best epsilon: 0.00301
Best K: 164
>> Test R^2 = 0.687516


In [25]:
searcher2  = tune_hyperparams(SVR(kernel="linear"), "svr_linear_kernel",
                           X_fit, y_fit, 
                           cv=10,
                           search="grid", params_space={"univariate_percentile__percentile": percentile_grid,
                                                       "svr_linear_kernel__C": c_grid,
                                                       "svr_linear_kernel__epsilon": epsilon_grid},
                            refit=False,
                            univariate_percentile=SelectPercentile(f_regression)
                            )
print("**10-Fold Cross-Validation w/ Best-Percentile Selector (F-Regression) Using Grid Search**")
print(f"""Best C: {round(searcher2.best_params_['svr_linear_kernel__C'], 6)}
Best epsilon: {round(searcher2.best_params_['svr_linear_kernel__epsilon'], 6)}
Best percentile: {searcher2.best_params_['univariate_percentile__percentile']}""")
pipe2 = Pipeline(steps=[
                       ("univariate_percentile", SelectPercentile(f_regression)),
                       ("svr_linear_kernel", SVR(kernel="linear"))     
                       ])
pipe2.set_params(**searcher2.best_params_)
pipe2.fit(X_fit, y_fit)
print(f">> Test R^2 = {round(pipe2.score(X_test, y_test), 6)}")

# CV search and fixed held-out set search are different validation objectives
# best_params results do not match is.... not abnormal
# It seems the best_params results are wildly different though 
# -> the hyperparameter choice probably unstable across selection protocols?

cv passed as an integer. Note that by default sklearn will NOT shuffle the folds. If your data are ordered (e.g., sorted by target value),
pass cv as a BaseCrossValidator object an set shuffle=True.
**10-Fold Cross-Validation w/ Best-Percentile Selector (F-Regression) Using Grid Search**
Best C: 0.297635
Best epsilon: 0.001
Best percentile: 24
>> Test R^2 = 0.594984


In [ ]:
def fisher(X):
    eps = 1e-8
    X = np.clip(X, -1+eps, 1-eps) # avoid zero division and zero ln
    return np.arctanh(X)
fisher_transformer = FunctionTransformer(fisher)

X_train_outer, X_test, y_train_outer, y_test = train_test_split(X, y,
                                                test_size=TEST_SIZE,
                                                shuffle=True,
                                                stratify=pheno["AgeGroup"],
                                                random_state=RANDOM_STATE)

kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

var_dist = 1e-2 * np.arange(75, 95)
results = nested_cv(X_train_outer, y_train_outer,
                    estimator_name="svr", estimator_params={"kernel": "rbf"},
                    inner_scoring=None, outer_scoring=mean_absolute_error,
                    stratify=False, n_repeats=None, outer_cv="loo",
                    inner_cv=kf, search="rand", params_space={#"univariate_best_k__k": k_dist,
                                                               "pca__n_components": var_dist,
                                                               "svr__C": c_dist,
                                                               "svr__epsilon": epsilon_dist,
                                                               "svr__gamma": gamma_dist},
                    fisher_transformer=fisher_transformer,
                    pca=PCA())
                    #univariate_best_k=SelectKBest(f_regression))

clear_output()

In [205]:
outer_cv_scores = np.array(results.outer_cv_scores)
print(f"Outer CV MAE mean = {outer_cv_scores.mean()}")
print(f"Outer CV MAE std = {outer_cv_scores.std()}")

aggregated_oof_r2 = r2_score(y_train_outer, results.outer_oof_predictions[1])
print(f"Aggregated CV out-of-fold R^2 = {aggregated_oof_r2}")

Outer CV MAE mean = 3.1977998119636184
Outer CV MAE std = 3.682487499548706
Aggregated CV out-of-fold R^2 = 0.6241602048428805


### Train/test sets
- Training set $D \sim P^n$, where each $(x, y) \in D$ is independently drawn from the same population distribution $P$.
- Test set input $X_{*} \sim P_X$, the marginal distribution of $X$ over the population distribution (i.e., the joint distribution of $X, Y$ summed over $Y$).
- Test set targets $Y_{*} \sim P_{Y\mid X}$ conditioned on $X = X_{*}$.<br><br>

### Model estimate
Suppose the real input-target function is $f$, then the data is generated by

$y_{*} = f(X_{*} = x_{*}) + \varepsilon_{*}$

where $\varepsilon$ is irreducible noise, assuming, for each test point $x_{*}$,

$\mathbb{E}_{\varepsilon_{*}}[\varepsilon_{*} \mid X_{*} = x_{*}] = 0$<br>
$\operatorname{Var}_{\varepsilon_{*}}[\varepsilon_{*} \mid X_{*} = x_{*}] = \sigma^2(x_{*})$

Given a training set $D$, the model estimates a function $\hat{f}_D(X)$, so the squared error for that test point is

$(y_{*} - \hat{f}_D(x_{*}))^2$

Taking the expected value over training sets $D$ and noise $\varepsilon$,

$\mathbb{E}_{D,\varepsilon_{*}}\left[(y_{*} - \hat{f}_D(x_{*}))^2\right]
= \mathbb{E}_{D,\varepsilon_{*}}\left[(f(x_{*}) + \varepsilon_{*} - \hat{f}_D(x_{*}))^2\right]$

Now define

$\bar{f}(.) = \mathbb{E}_D[\hat{f}_D(.)]$

meaning the expected model estimation of the function $f$ over randomly sampled training sets. Then

$f(x_{*}) - \bar{f}(x_{*})$

measures, intuitively, "<i>how far away from the 'truth' the model estimate is for test point $x_{*}$ if we repeatedly sample training sets and average the model predictions?</i>"<br><br>

### Bias-variance tradeoff
Plugging this term in and expanding it carefully, we have

$[f(x_{*}) - \bar{f}(x_{*}) + \bar{f}(x_{*}) - \hat{f}_D(x_{*}) + \varepsilon_{*}]^2$<br>
$= [f(x_{*}) - \bar{f}(x_{*})]^2 + [\bar{f}(x_{*}) - \hat{f}_{D}(x_{*})]^2 + \varepsilon_{*}^2$<br>
$+ 2[f(x_{*}) - \bar{f}(x_{*})][\bar{f}(x_{*}) - \hat{f}_D(x_{*})]$<br>
$+ 2[f(x_{*}) - \bar{f}(x_{*})]\varepsilon_{*}$<br>
$+ 2[\bar{f}(x_{*}) - \hat{f}_D(x_{*})]\varepsilon_{*}$

Notice all three cross terms vanish in expected values calculation:
> <strong>1st cross term</strong><br><br>
$\mathbb{E}_{D,\varepsilon_{*}}\left[[f(x_{*}) - \bar{f}(x_{*})][\bar{f}(x_{*}) - \hat{f}_D(x_{*})]\right]$<br>
$= \mathbb{E}_{D,\varepsilon_{*}}\left[f(x_{*})\bar{f}(x_{*}) - f(x_{*})\hat{f}_D(x_{*}) - \bar{f}(x_{*})\bar{f}(x_{*}) + \bar{f}(x_{*}) \hat{f}_D(x_{*})\right]$<br>
$= \mathbb{E}_{D,\varepsilon_{*}}\left[f(x_{*})\bar{f}(x_{*})\right] - \mathbb{E}_{D,\varepsilon_{*}}\left[f(x_{*})\hat{f}_D(x_{*})\right] - \mathbb{E}_{D,\varepsilon_{*}}\left[\bar{f}(x_{*})\bar{f}(x_{*})\right] + \mathbb{E}_{D,\varepsilon_{*}}\left[\bar{f}(x_{*}) \hat{f}_D(x_{*})\right]$<br>
$= f(x_{*})\bar{f}(x_{*}) - f(x_{*})\mathbb{E}_{D,\varepsilon_{*}}[\hat{f}_D(x_{*})] - \bar{f}(x_{*})\bar{f}(x_{*}) + \bar{f}(x_{*})\mathbb{E}_{D,\varepsilon_{*}}\left[\hat{f}_D(x_{*})\right]$<br>
$= f(x_{*})\bar{f}(x_{*}) - f(x_{*})\bar{f}(x_{*}) + \bar{f}(x_{*})\bar{f}(x_{*}) - \bar{f}(x_{*})\bar{f}(x_{*}) = 0$

since

$\mathbb{E}_{D,\varepsilon_{*}}\left[\hat{f}_D(x_{*})\right] = \mathbb{E}_{D}\left[\hat{f}_D(x_{*})\right] = \bar{f}(x_{*})$

as it is independent of $\varepsilon_{*}$.

> <strong>2nd cross term</strong><br><br>
$\mathbb{E}_{D,\varepsilon_{*}}\left[[f(x_{*}) - \bar{f}(x_{*})]\varepsilon_{*}\right] = \mathbb{E}_{D}\left[f(x_{*}) - \bar{f}(x_{*})\right]\mathbb{E}_{\varepsilon_{*}}\left[\varepsilon_{*}\right] = 0$

And likewise,

> <strong>3rd cross term</strong><br><br>
$\mathbb{E}_{D,\varepsilon_{*}}\left[[\bar{f}(x_{*}) - \hat{f}_D(x_{*})]\varepsilon_{*}\right] = 0$<br><br>

So the remaing parts (squared terms):

$\mathbb{E}_{D,\varepsilon_{*}}\left[(y_{*} - \hat{f}_D(x_{*}))^2\right]$<br>
$=\mathbb{E}_{D,\varepsilon_{*}}\left[(f(x_{*}) - \bar{f}(x_{*}))^2\right]$
$+ \mathbb{E}_{D,\varepsilon_{*}}\left[(\bar{f}(x_{*}) - \hat{f}_D(x_{*}))^2\right]$
$+ \mathbb{E}_{D,\varepsilon_{*}}\left[\varepsilon_{*}^2\right]$<br>
$= (f(x_{*}) - \bar{f}(x_{*}))^2 + \mathbb{E}_{D}\left[(\bar{f}(x_{*}) - \hat{f}_D(x_{*}))^2\right] + \operatorname{Var}[\varepsilon_{*} \mid X_{*} = x_{*}]$<br>
$= (f(x_{*}) - \bar{f}(x_{*}))^2 + \mathbb{E}_{D}\left[(\bar{f}(x_{*}) - \hat{f}_D(x_{*}))^2\right] + \sigma^2(x_{*})$

conditional on a single given test point $x_{*}$<br><br>

Now we take the expected value over $X_{*} \sim P_X$ to get the total MSE,

$\mathbb{E}_{X_{*}}\left[\mathbb{E}_{D,\varepsilon_{*}}\left[(Y_{*} - \hat{f}_D(X_{*}))^2\right] \mid X_{*}\right]$<br>
$= \mathbb{E}_{X_{*},D,\varepsilon}\left[(Y_{*} - \hat{f}_D(X_{*}))^2\right]$<br>
$= \mathbb{E}_{X_{*}}\left[(f(X_{*}) - \bar{f}(X_{*}))^2\right] + \mathbb{E}_{X_{*},D}\left[(\bar{f}(X_{*}) - \hat{f}_D(X_{*}))^2\right] + \mathbb{E}_{X_{*}}\left[\operatorname{Var}\left[\varepsilon_{*}\mid X_{*}\right]\right]$

Where
- Bias (squared): $\mathbb{E}_{X_{*}}\left[(f(X_{*}) - \bar{f}(X_{*}))^2\right]$  
- Variance:   $\mathbb{E}_{X_{*},D}\left[(\bar{f}(X_{*}) - \hat{f}_D(X_{*}))^2\right]$
- Irreducible noise:   $\mathbb{E}_{X_{*}}\left[\operatorname{Var}\left[\varepsilon_{*}\mid X_{*}\right]\right] = \mathbb{E}_{X_{*}}\sigma^2(X_{*})$

### Example: OLS Regression
In OLS regression, we use the linear model

$\mathbf{y} = X{\boldsymbol{\beta}} + {\boldsymbol{\varepsilon}}$<br>

where<br>
- $\mathbf{y} \in \mathbb{R}^n$
- $X \in \mathbb{R}^{n \times p}$
- ${\boldsymbol{\beta}} \in \mathbb{R}^p$
- ${\boldsymbol{\varepsilon}} \in \mathbb{R}^n$<br>

And we also assume:
- $\mathbb{E}\left[{\boldsymbol{\varepsilon}} \mid X\right] = {\boldsymbol{0}}$
- $\operatorname{Var}\left({\boldsymbol{\varepsilon}} \mid X\right) = \sigma^2 I$

The closed -form solution is

$\hat{\boldsymbol{\beta}} = (X^\top X)^{-1} X^\top \mathbf{y}$

Assuming $(X^\top X)^{-1}$ is invertible.<br><br>

### Expected Value and Variance of the Model Estimate $\hat{\boldsymbol{\beta}}$ (conditional on design matrix $X$)
#### 1. Expected Value
$\mathbb{E}\left[\hat{\boldsymbol{\beta}} \mid X\right]$<br>
$= \mathbb{E}\left[(X^\top X)^{-1} X^\top \mathbf{y} \mid X\right]$<br>
$= \mathbb{E}\left[(X^\top X)^{-1} X^\top (X{\boldsymbol{\beta}} + {\boldsymbol{\epsilon}}) \mid X\right]$<br>
$= \mathbb{E}\left[I{\boldsymbol{\beta}} + (X^\top X)^{-1} X^\top {\boldsymbol{\varepsilon}} \mid X\right]$<br>
$= \mathbb{E}\left[{\boldsymbol{\beta}} \mid X \right] + (X^\top X)^{-1} X^\top \mathbb{E}\left[{\boldsymbol{\varepsilon}} \mid X \right] $<br>
$= {\boldsymbol{\beta}}$<br>

Since $X$ and ${\boldsymbol{\beta}}$ are fixed and $\mathbb{E}\left[{\boldsymbol{\varepsilon}} \mid X\right] = {\boldsymbol{0}}$. That is, $\hat{\boldsymbol{\beta}}$ is an <i>unbiased</i> estimate of ${\boldsymbol{\beta}}$ conditioned on $X$.<br>

#### 2. Variance
$\operatorname{Var}\left(\hat{\boldsymbol{\beta}} \mid X\right)$<br>
$= \operatorname{Var}\left((X^\top X)^{-1} X^\top (X{\boldsymbol{\beta}} + {\boldsymbol{\varepsilon}}) \mid X\right)$<br>
$= \operatorname{Var}\left(I{\boldsymbol{\beta}} + (X^\top X)^{-1} X^\top {\boldsymbol{\varepsilon}} \mid X\right)$<br>
$= \operatorname{Var}\left((X^\top X)^{-1} X^\top {\boldsymbol{\varepsilon}} \mid X\right)$

Since variance is invariant to the addition/subtraction of a constant. Then, using the <i>matrix rule</i>, i.e., 

$\operatorname{Var}\left(AZ\right)$<br>
$= (AZ - {\boldsymbol{\mu_{AZ}}})(AZ - {\boldsymbol{\mu_{AZ}}})^\top$<br>
$= (A(Z - {\boldsymbol{\mu_{Z}}}))(A(Z - {\boldsymbol{\mu_{Z}}}))^\top$<br>
$= A(Z - {\boldsymbol{\mu_{Z}}})(Z - {\boldsymbol{\mu_{Z}}})^\top A^\top$<br>
$= A\operatorname{Var}\left(Z\right)A^\top$

We have

$= \operatorname{Var}\left((X^\top X)^{-1} X^\top {\boldsymbol{\varepsilon}} \mid X\right)$<br>
$= (X^\top X)^{-1} X^\top \operatorname{Var}\left( {\boldsymbol{\varepsilon}} \mid X\right) X ((X^\top X)^{-1})^\top$<br>
$= (X^\top X)^{-1} X^\top \sigma^2 I X ((X^\top X)^{-1})^\top$<br>
$= \sigma^2 (X^\top X)^{-1} X^\top X ((X^\top X)^{-1})^\top$<br>
$= \sigma ^2 ((X^\top X)^{-1})^\top$<br>
$= \sigma^2 (X^\top X)^{-1}$

Since $X^\top X$ is symmetric, we can factor it into its eigen value deconposition (see [Math Crash Course](#Math-Crash-Course) below):

$X^\top X = Q \Lambda Q^\top$

where

$\Lambda = \operatorname{diag}(\lambda_1, \lambda_2, \dots, \lambda_p), \lambda_1 \geq \lambda_2 \geq \dots, \geq \lambda_p > 0$ (assuming invertibility), so

$(X^\top X )^{-1} = Q \Lambda^{-1} Q^\top$, $((X^\top X)^{-1})^\top = Q\Lambda^{-1} Q^\top = (X^\top X )^{-1}$.

Thus, the variance can be expressed in terms of the eigenvalues:

$\operatorname{Var}\left(\hat{\boldsymbol{\beta}} \mid X\right) = \sigma^2 Q \Lambda^{-1} Q^\top$<br>

#### 3. Variance of <i>the Predition</i>
Note that in the definition of the bias-variance trade-off, model variance refers to variances in model predictions. Below we compute prediction variance.

$\operatorname{Var}(\hat{f}(x_*) \mid X) = \operatorname{Var}(x_*^\top \hat{\beta} \mid X) = x_*^\top \operatorname{Var} (\hat{\beta} \mid X) x_* = \sigma^2 x_*^\top Q \Lambda^{-1} Q^\top x_*$

> <strong>NOTE</strong> that in the above derivation of the bias-variance trade-off, variance is across all possible training sets $D \sim P^n$, whereas in the OLS variance just derived, variance is conditional <i>on a fixed, given training set</i>.

### Variance and <strong>Overfitting</strong>
#### Math Crash Course
First, we review some linear algebra facts. Below we assume $X$ is an $n \times p$ real matrix (so $X ^\top X$ is a $p \times p$ square matrix).<br><br>

##### 1. $X^\top X$ is <i>positive semidefinite</I>
> <strong>Definition</strong><br>A matrix $A$ is positive semidefinite if $\forall v \neq {\boldsymbol{0}}, v^\top Av \geq 0$.

So by definition,

$v^\top X^\top Xv = (Xv)^\top (Xv) = \lVert Xv \rVert \geq 0$.

If $rank(X) = p$, i.e., full column rank, then $Xv = {\boldsymbol{0}} \Leftrightarrow v = {\boldsymbol{0}}$, and $X^\top X$ is positive <i>definite</i>. If $rank(X) < p$, then $\exists v \neq {\boldsymbol{0}}$ such that $Xv = {\boldsymbol{0}}$, i.e., $v \in ker(X)$.<br><br>

##### 2. $rank(X^\top X) = rank(X)$
> <strong>Proof</strong><br>Let $v \in ker(X^\top X)$, then $\lVert Xv \rVert = (Xv)^\top (Xv) = v^\top X^\top Xv = v(X^\top X)v = {\boldsymbol{0}}$<br>$\Rightarrow Xv = {\boldsymbol{0}}$<br>$\Rightarrow v \in ker(X)$<br>$\Rightarrow ker(X^\top X) \subseteq ker(X)$<br><br>The converse, i.e., $ker(X) \subseteq ker(X^\top X)$ is trivial.

Thus, by the rank-nullity theorem, $rank(X^\top X) = p - dim(ker(X^\top X)) = n - dim(ker(X)) = rank(X)$.<br><br>

##### 3. $X^\top X$ is <i>diagonalizable</i>.
First we introduce the Spectral Theorem:
> <strong>Theorem</strong><br>If $A$ is a Hermitian matrix (i.e., a matrix that is equal to its conjugate transpose) over a finite-dimensional inner-product space $V$, then there exists an orthonormal basis of $V$ consisting of eigen vectors of $A$, and each eigenvalue of $A$ is real.

Since $X$ is real, and since $(X^\top X)^\top = X^\top X$, $X\top X$ is symmetric, hence Hermitian (in a restrictive sense). So there exists an orthonormal basis of $\mathbb{R}^p$ consisting of eigen vectors of $X^\top X$.

Moreover, a linear map $T: V \longrightarrow V$ is <i>diagonalizable</i> if and only if the sum of the dimensions of the eigen spaces of $T$ equals $dim(V)$. Since $X^\top X$ has an orthonormal eigen basis for $\mathbb{R}^p$, $X^\top X$ is diagonalizable (possibly with zero eigen values).<br><br>

##### 4. $det(X^\top X) = \prod_{i}\lambda_{i}$
Since $X^\top X$ is diagonalizable, write

$X^\top X = Q \Lambda Q^\top$

where

$\Lambda = \operatorname{diag}(\lambda_1, \lambda_2, \dots, \lambda_p), \lambda_1 \geq \lambda_2 \geq \dots, \geq \lambda_p \geq 0$

and $Q$ is the matrix whose $i$th column is the unit (normalized) eigen vector corresponding to $\lambda_i$. Since the eigen vectors form an orthonormal basis of $\mathbb{R}^p$, $Q^\top Q = I$, so $Q^\top = Q^{-1}$.<br>

$det(X^\top X) = det(Q \Lambda Q^\top) = det(Q)det(\Lambda)det(Q^\top) = det(\Lambda)det(Q)det(Q^\top) = det(\Lambda)det(QQ^\top) = det(\Lambda) = \prod_{i}\lambda_{i}$<br>

Moreover, $X^\top X$ is invertible $\Leftrightarrow det(X^\top X) \neq 0$, we have the following result:

> $X$ is rank deficient ($rank(X) < p$) $\Leftrightarrow X^\top X$ is singular (i.e., non-invertible) $\Leftrightarrow X^\top X$ has at least one zero eigen value.

<br>

##### 5. The Rayleigh quotient and "near-singularity"
> <strong>Definition</strong><br>The Rayleigh quotient for a given hermitian matrix $A$ and nonzero vector $v$ is defined as<br><br>$R(A,v) = \frac{v^{*}Av}{v^{*}v}$, where $x^{*}$ denotes the conjugate tranpose.<br><br>For real matrices, the condition Hermitian is reduced to that of being symmetric (as in the Spectral Theorem), and the conjugate transpose is reduced to the transpose $x^\top$.

It can be shown that, for any given Hermitian matrix $A$,

$\lambda_{min} \leq R(A,v) \leq \lambda_{max}$,

where $\lambda_{min}$ and $\lambda_{max}$ are the smallest and largest eigenvalues, respectively, and

$R(A, v_{min}) = \lambda_{min}$, $R(A, v_{max}) = \lambda_{max}$,

where $v_{min}$ and $v_{max}$ are the eigen vectors corresponding to $\lambda_{min}$ and $\lambda_{max}$, respectively. <sub>[1]</sub>

> <strong>Proof</strong><br>Again write $X^\top X = Q \Lambda Q^\top$, and fix ${\lVert v \rVert} = 1$.<br><br>Since $Q = [q_1, ... q_p]$ forms an orthonormal basis of $\mathbb{R}^p$, we can write $v$ as its linear combination $v = \sum_{i}\alpha_{i} q_{i}$, with $\sum_{i} \alpha_{i}^2 = 1$.<br><br>$v^\top Q \Lambda Q^\top v = \sum_{i}\lambda_{i}v^\top q_{i}q_{i}^\top v = \sum_{i}\lambda_{i}\lVert q_{i}^\top v \rVert^2 = \sum_{i,j}\lambda_{i} \lVert q_{i}^\top \alpha_{j}q_{j} \rVert^2 = \sum_{i,j}\lambda_{i} \alpha_{j}^2\delta_{i,j} = \sum_{i} \alpha_{i}^2 \lambda_{i}$ Given that $\sum_{i} \alpha_{i}^2 = 1$, the sum's maximum and minimum is $\lambda_{max}$ and $\lambda_{min}$, respectively

<sub>[1]: This is actually the core of <strong>PCA (principal component analysis)</strong>.</sub>

Now consider the case where columns of $X$ are "nearly linearly <i>dependent</i>".

From the perspective of linear combinations, this means that there exists coefficients $ v \neq {\boldsymbol{0}}$ such that $Xv \approx {\boldsymbol{0}}$. If we scale the coefficients such that ${\lVert v \rVert} = 1$, it follows that the Rayleigh quotient

$R(X^\top X,v) = \frac{v^\top XX^\top v}{v^\top v} = \frac{\lVert Xv \rVert}{\lVert v \rVert} \approx 0$

This means that the smallest eigenvalue of $X^\top X$ must also be very close to zero.<br><br>

#### Multi-collinearity $\longrightarrow$ small eigenvalues $\longrightarrow$ variance explosion

### Regularization: bias injection and variance reduction
- Ridge regression as an example
- Sample size matters

### Bias-variance trade-off: underfitting and overfitting — the empirical aspects